In [1]:
import mne
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

mne.set_log_level('WARNING')
print("imports ok")

imports ok


In [2]:
# load raw data
raw = mne.io.read_raw_gdf('../data/A01T.gdf', preload=True)

# apply bandpass filter 8-30 Hz
raw.filter(l_freq=8, h_freq=30, picks='eeg', method='iir')

# extract events
events, event_id = mne.events_from_annotations(raw)

# define motor imagery classes
event_id_mi = {
    'left_hand':  7,
    'right_hand': 8,
    'feet':       9,
    'tongue':     10
}

# extract epochs
epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id_mi,
    tmin=0.0,
    tmax=4.0,
    baseline=None,
    preload=True,
    reject=dict(eeg=100e-6)
)

print(f"Epochs loaded: {epochs.get_data().shape}")

c:\Python314\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Epochs loaded: (288, 25, 1001)


In [3]:
# drop EOG channels — we only want brain signals
epochs.drop_channels(['EOG-left', 'EOG-central', 'EOG-right'])

# get the data as numpy array
X = epochs.get_data()  # shape: (288, 22, 1001)
y = epochs.events[:, 2] - 7  # convert labels 7,8,9,10 → 0,1,2,3

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Labels: {np.unique(y)} → {['left_hand', 'right_hand', 'feet', 'tongue']}")
print(f"Samples per class: {[(y==i).sum() for i in range(4)]}")

X shape: (288, 22, 1001)
y shape: (288,)
Labels: [0 1 2 3] → ['left_hand', 'right_hand', 'feet', 'tongue']
Samples per class: [np.int64(72), np.int64(72), np.int64(72), np.int64(72)]


In [4]:
# normalize each channel independently across all trials
# X shape: (288, 22, 1001)
X_normalized = np.zeros_like(X)

for ch in range(X.shape[1]):
    # get all trials for this channel
    ch_data = X[:, ch, :]  # shape: (288, 1001)
    
    # compute mean and std across all trials and timepoints
    mean = ch_data.mean()
    std = ch_data.std()
    
    # normalize
    X_normalized[:, ch, :] = (ch_data - mean) / std

print(f"Before normalization:")
print(f"  Mean: {X.mean():.6f}, Std: {X.std():.6f}")
print(f"\nAfter normalization:")
print(f"  Mean: {X_normalized.mean():.6f}, Std: {X_normalized.std():.6f}")
print(f"\nData ready for EEGNet")

Before normalization:
  Mean: -0.000000, Std: 0.000005

After normalization:
  Mean: 0.000000, Std: 1.000000

Data ready for EEGNet


In [5]:
from sklearn.model_selection import train_test_split

# split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keep class balance in both splits
)

print(f"Training set: {X_train.shape} — {len(y_train)} trials")
print(f"Test set:     {X_test.shape} — {len(y_test)} trials")
print(f"\nClass distribution in training set:")
for i, name in enumerate(['left_hand', 'right_hand', 'feet', 'tongue']):
    print(f"  {name}: {(y_train==i).sum()} trials")

# save to disk
np.savez('../data/preprocessed.npz',
         X_train=X_train,
         X_test=X_test,
         y_train=y_train,
         y_test=y_test)

print(f"\nSaved to data/preprocessed.npz")

Training set: (230, 22, 1001) — 230 trials
Test set:     (58, 22, 1001) — 58 trials

Class distribution in training set:
  left_hand: 57 trials
  right_hand: 58 trials
  feet: 57 trials
  tongue: 58 trials

Saved to data/preprocessed.npz
